In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv


In [43]:
INPUT_DIR = "/kaggle/input/deep-past-initiative-machine-translation/"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")
publications_df = pd.read_csv(f"{INPUT_DIR}/publications.csv")
dict_link_df = pd.read_csv(f"{INPUT_DIR}/OA_Lexicon_eBL.csv")
dict_df = pd.read_csv(f"{INPUT_DIR}/eBL_Dictionary.csv")
alignment_df = pd.read_csv(f"{INPUT_DIR}/Sentences_Oare_FirstWord_LinNum.csv")
resources_df = pd.read_csv(f"{INPUT_DIR}/resources.csv")
published_df = pd.read_csv(f"{INPUT_DIR}/published_texts.csv")

Remove all gaps and treat them as seperate sentences.

use train_df, test_df, published_df as data for the word2vec model.
If there's a way I can get the counts of each word I would like to know.

https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html

In [44]:
# For loop through train_df, test_df, published_df transliterations all as one big list.
# Add each sentence to the thing. 
# If a sentence has one or more gaps/big gaps (<big_gap>, <gap>), split on those

sentences = []
for df in [train_df, test_df, published_df]:
    for col in ["transliteration"]:
        for sentence in df[col].dropna():
            sentences.append(sentence)
            if "<big_gap>" in sentence:
                sentences.extend(sentence.split("<big_gap>"))
            if "<gap>" in sentence:
                sentences.extend(sentence.split("<gap>"))
sentences = list(set(sentences))  # Remove duplicates
sentences = [s.strip() for s in sentences if s.strip()]  # Remove empty


print(len(sentences))
print(sentences[:10])

31123
['1 GÚ URUDU SIG5', 'KIŠIB {d}en-LÍL-ba-ni DUMU GAL-a-šur KIŠIB LUGAL-{d}IŠKUR DUMU nu-ur-sú 12 GÍN KÙ.BABBAR ṣa-ru-pá-am i-ṣé-er LUGAL-{d}IŠKUR DUMU nu-ur-sú DAM.GÀR i-šu iš-tù ḫa-mu-uš-tim ša ḫa-na-a ITU.KAM ku-zal-li li-mu-um e-na-sú-en6 DUMU šu-a-šur a-na 8 ḫa-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i-na ITU.KAM ṣí-ib-tám ú-ṣa-áb', 'i-na lu-qú-ut 10 ma-na KÙ.BABBAR ša a-šùr-SIPA ù a-šùr-be-el-ma-al-ki-im ša iš-tù a-lim{ki} e-li-a-ni a-na 6 ma-na KÙ.BABBAR a-šùr-SIPA i-la-qé a-na 4 ma-na KÙ.BABBAR a-šùr-be-el-ma-al-ki-im i-lá-qé a-na 1 GÚ-tim AN.NA ú li-wi-tí-šu ša i-na mì-da-áš-ku-ri-a ḫa-al-qú-ni 12 ma-na AN.NA i qá-tí a-šur-be-el-ma-al-ki-im i-ṣa-ḫe-er li-wi-tum a šu-mì-šu-nu-ma i-ḫa-li-iq ší-tí lu-qú-tí-šu-nu-tí a-na ba-a-ba-at a-wi-it-šu-nu i-zu-zu 45 ma-na AN.NA ku-nu-ki ša a-de8-lá-at ù a-šur-ba-ni <gap> a-šur-be-el-ma-al-ki-im ú-ša-za-az-ma i-lá-qé 1 ma-na 10 GÍN ša iš-tí DUMU ú-zu-a a-šur-SIPA il5-qé-ú ša a-šur-SIPA-ma a-na a-wa-tim a-ni-a-tim 

In [45]:
from gensim.test.utils import datapath
from gensim import utils

class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __iter__(self):
        # corpus_path = datapath('lee_background.cor')
        # for line in open(corpus_path):
        for line in sentences:
            # assume there's one document per line, tokens separated by whitespace
            yield utils.simple_tokenize(line)

In [46]:
import gensim.models

sentencesCorpus = MyCorpus()
model = gensim.models.Word2Vec(sentences=sentencesCorpus, min_count=1, vector_size=10, epochs=10, workers=1)

TypeError: object of type 'generator' has no len()

In [ ]:
for index, word in enumerate(model.wv.index_to_key):
    if index == 10:
        break
    print(f"word #{index}/{len(model.wv.index_to_key)} is {word}")

In [ ]:
from sklearn.decomposition import IncrementalPCA    # inital reduction
from sklearn.manifold import TSNE                   # final reduction
import numpy as np                                  # array handling


def reduce_dimensions(model):
    num_dimensions = 2  # final num dimensions (2D, 3D, etc)

    # extract the words & their vectors, as numpy arrays
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

    # reduce using t-SNE
    # tsne = TSNE(n_components=num_dimensions, random_state=0)
    # vectors = tsne.fit_transform(vectors)
    pca = IncrementalPCA(n_components=num_dimensions)
    vectors = pca.fit_transform(vectors)

    x_vals = [v[0] for v in vectors]
    y_vals = [v[1] for v in vectors]
    return x_vals, y_vals, labels


x_vals, y_vals, labels = reduce_dimensions(model)

def plot_with_plotly(x_vals, y_vals, labels, plot_in_notebook=True):
    from plotly.offline import init_notebook_mode, iplot, plot
    import plotly.graph_objs as go

    trace = go.Scatter(x=x_vals, y=y_vals, mode='text', text=labels)
    data = [trace]

    if plot_in_notebook:
        init_notebook_mode(connected=True)
        iplot(data, filename='word-embedding-plot')
    else:
        plot(data, filename='word-embedding-plot.html')


def plot_with_matplotlib(x_vals, y_vals, labels):
    import matplotlib.pyplot as plt
    import random

    random.seed(0)

    plt.figure(figsize=(12, 12))
    plt.scatter(x_vals, y_vals)

    #
    # Label randomly subsampled 25 data points
    #
    indices = list(range(len(labels)))
    selected_indices = random.sample(indices, 504)
    for i in selected_indices:
        plt.annotate(labels[i], (x_vals[i], y_vals[i]))
    plt.show()

# try:
    # get_ipython()
# except Exception:
    print("A")
plot_function = plot_with_matplotlib
# else:
    # print("B")
    # plot_function = plot_with_plotly

plot_function(x_vals, y_vals, labels)

In [ ]:
import tempfile

with tempfile.NamedTemporaryFile(prefix='gensim-model-', delete=False) as tmp:
    temporary_filepath = tmp.name
    model.save(temporary_filepath)
    #
    # The model is now safely stored in the filepath.
    # You can copy it to other machines, share it with others, etc.
    #
    # To load a saved model:
    #
    new_model = gensim.models.Word2Vec.load(temporary_filepath)